# HomeReach Melbourne — 04: Match LGA boundaries and map rents

**One question:** where are the different published LGA rents located?

**Rent snapshot:** 2025Q3, `2 bedroom flats`, the existing 31 source-group metropolitan LGAs.
**Boundary snapshot:** the uploaded `LGA_2025_AUST_GDA2020.zip` (2025 ABS statistical LGA boundaries).

Run with your existing **homereach** kernel, from top to bottom, using the cell ▶ or Ctrl+Enter.
Keep the ZIP unchanged in `data/raw/geography/`; no manual extraction or new environment is needed.
This notebook reads the rental pipeline's validated CSV. It does not rerun cleaning or depend on another notebook's memory.

### What you will learn
- Inspect the archive and distinguish spatial LGAs from non-spatial statistical records.
- Build an auditable **crosswalk** (a table mapping source names to boundary codes).
- Make an **attribute join**, not a spatial join or fuzzy-name guess.
- Validate that all 31 areas, all original rental fields, and the correct geometries survive.
- Display one interactive choropleth: colour represents median **AUD per week**, not suitability.

### Interpretation boundary
ABS boundaries are statistical approximations, not legal boundaries. Matching names/codes is a
reviewed association for this prototype, not proof that the rental publisher used identical polygons.
Keep 2025 boundaries with the 2025Q3 map for now; this does not harmonise historical boundaries.
The source's metropolitan groups remain our scope—not all Victorian polygons and not a newly substituted
ABS Greater Capital City Statistical Area definition. No transport, routing, forecast, or recommendation is added.

Sources: ABS LGA descriptions and coding rules:
https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-3-july-2021-june-2026/non-abs-structures/local-government-areas

DFFH rent definitions: https://www.dffh.vic.gov.au/publications/rental-report

**Inputs remain unchanged.** Generated outputs are limited to `data/interim/geography/`,
`data/processed/geography/metro_lga_2025.gpkg`, and `docs/lga_geography_audit.md`.
The registry, notebooks 01–03, and Streamlit demonstration are not edited.

## 1. Imports and paths

A GeoDataFrame is a table with a `geometry` column and a coordinate reference system (CRS).
`pyogrio` reads the geographic layer using GDAL; `zipfile` lets us inspect the archive without extracting it.
We reuse `read_panel_csv()` from the already-tested rental module.

No package installation occurs in this notebook. The map uses Plotly's MapLibre-based API introduced in
5.24; the first cell checks that this API is available rather than silently using a deprecated substitute.

In [ ]:
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
from tempfile import TemporaryDirectory
from datetime import datetime, timezone
import json
import sys
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio
import pyproj
import shapely
import plotly
import plotly.graph_objects as go
from IPython.display import display
from homereach.io.dffh_rent import (
    COLUMN_ORDER, KEY_COLUMNS, AGGREGATE_LABELS,
    read_panel_csv, load_snapshot, fingerprint, require,
)

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 40)
pd.set_option("display.max_colwidth", 65)

def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook inside your existing HomeReach project.")

ROOT = find_project_root(Path.cwd())
INPUT = ROOT / "data/processed/rent_quarterly.csv"
SPEC_PATH = ROOT / "config/rent_snapshot.json"
QUALITY_PATH = ROOT / "data/processed/rent_quality_report.json"
BOUNDARY_ZIP = ROOT / "data/raw/geography/LGA_2025_AUST_GDA2020.zip"
EXPECTED_ZIP_SHA256 = "acc3015a0ac78ade978c41a2e4110269b5219b60d6a56cbb70393ae647f31b15"
BOUNDARY_YEAR = 2025
LAYER = "LGA_2025_AUST_GDA2020"
ABS_PAGE = (
    "https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/"
    "edition-3-july-2021-june-2026/non-abs-structures/local-government-areas"
)
ABS_DOWNLOAD = (
    "https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/"
    "edition-3-july-2021-june-2026/access-and-downloads/digital-boundary-files/"
    "LGA_2025_AUST_GDA2020.zip"
)
for path in [INPUT, SPEC_PATH, QUALITY_PATH, BOUNDARY_ZIP]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path.relative_to(ROOT)}. Restore the actual input, not a placeholder.")
require(hasattr(go, "Choroplethmap"),
        "This map needs Plotly 5.24 or newer. Report the installed version; do not recreate the environment.")
print("Python:", sys.executable)
print("pandas:", pd.__version__, "| GeoPandas:", gpd.__version__, "| Plotly:", plotly.__version__)
print("Project root:", ROOT)

## 2. Read the existing rental table and choose one quarter

We check the CSV fingerprint against its pipeline report, then select **2025Q3 only**.
The native source table still has 3,286 rows; the first map needs exactly **31 rows**.
The date check uses `as_unit("ns")` on both sides to avoid the pandas 2/3 timestamp-resolution issue.

A file fingerprint identifies this exact local snapshot. It is not an independent certification of its publisher.
Do not bypass a fingerprint mismatch; investigate the changed input.

In [ ]:
spec = load_snapshot(SPEC_PATH)
quality_report = json.loads(QUALITY_PATH.read_text(encoding="utf-8"))
require(quality_report.get("source", {}).get("sha256") == spec["sha256"],
        "The quality report belongs to a different rental snapshot.")
require(quality_report.get("output_sha256", {}).get(INPUT.name) == fingerprint(INPUT),
        "The rental CSV differs from its pipeline quality report.")
require(fingerprint(BOUNDARY_ZIP) == EXPECTED_ZIP_SHA256,
        "Boundary ZIP differs from the inspected upload. Re-audit it; do not change the expected hash to bypass this.")
panel = read_panel_csv(INPUT)
require(panel.columns.tolist() == COLUMN_ORDER, "Unexpected rental schema.")
require(panel[KEY_COLUMNS].notna().all().all() and not panel.duplicated(KEY_COLUMNS).any(),
        "Missing or duplicate rental key.")
require(not panel["rental_area_name"].isin(AGGREGATE_LABELS).any(), "Aggregate record in rental panel.")
for column, expected in {
    "area_type": "LGA", "property_type": spec["property_type"],
    "source_file": spec["filename"], "source_sheet": spec["sheet"],
    "source_reporting_quarter": spec["reporting_quarter"],
}.items():
    require(panel[column].notna().all() and panel[column].eq(expected).all(), f"Unexpected {column}.")
expected_quarters = pd.period_range(spec["first_quarter"], spec["last_quarter"], freq="Q")
expected_lgas = sum(spec["metro_group_counts"].values())
require(expected_lgas == 31 and len(panel) == 3286, "Reviewed rental scope changed.")
for key, group in panel.groupby(["rental_area_name", "property_type"], sort=False):
    require(group.sort_values("quarter_end")["quarter"].tolist() == expected_quarters.astype(str).tolist(),
            f"Incomplete quarter sequence for {key}.")
actual_dates = pd.DatetimeIndex(panel["quarter_end"]).as_unit("ns")
expected_dates = pd.PeriodIndex(panel["quarter"], freq="Q").to_timestamp(how="end").normalize().as_unit("ns")
require(np.array_equal(actual_dates.asi8, expected_dates.asi8), "Quarter labels and end dates disagree.")
for column in ["median_weekly_rent", "published_count"]:
    values = panel[column].to_numpy(dtype=float)
    require(np.isfinite(values).all() and (values > 0).all(), f"Invalid {column} in the reviewed snapshot.")

MAP_QUARTER = "2025Q3"
require(spec["last_quarter"] == MAP_QUARTER, "This notebook is reviewed for 2025Q3 only.")
latest = panel.loc[panel["quarter"].eq(MAP_QUARTER)].copy().reset_index(drop=True)
require(len(latest) == expected_lgas and latest["rental_area_name"].is_unique, "Latest-quarter inventory changed.")
require(latest.groupby("source_region").size().to_dict() == spec["metro_group_counts"], "Metro group scope changed.")

tracked_paths = [INPUT, SPEC_PATH, QUALITY_PATH, BOUNDARY_ZIP]
optional_paths = [ROOT / "data/raw/rent" / spec["filename"],
                  ROOT / "data/processed/rent_quarterly.parquet",
                  ROOT / "docs/data_registry.csv", ROOT / "docs/project_charter.md",
                  ROOT / "notebooks/01_data_audit.ipynb", ROOT / "notebooks/02_clean_rental_data.ipynb",
                  ROOT / "notebooks/03_rental_eda.ipynb"]
tracked_paths += [path for path in optional_paths if path.is_file()]
input_hashes = {path: fingerprint(path) for path in tracked_paths}
panel_before = panel.copy(deep=True)
latest_before = latest.copy(deep=True)
print("RENT INPUT CHECKPOINT PASSED")
print("Cleaned input rows:", len(panel))
print("Map quarter:", MAP_QUARTER, "| Rental rows to map:", len(latest))
print("Boundary ZIP matches inspected upload: True")

## 3. Inspect the ZIP and its metadata

A shapefile is not just one file: `.shp` stores shapes, `.dbf` stores attributes,
`.shx` is an index, `.prj` describes the CRS, `.cpg` describes text encoding, and
this archive also contains an ABS metadata `.xml` file. Keep them together inside the original ZIP.

We check the archive CRCs and read the metadata's title/licence. The 2025 archive has six members.
The metadata states CC BY 4.0 unless otherwise noted; attribution is retained in the exported report/map.

In [ ]:
with ZipFile(BOUNDARY_ZIP) as archive:
    require(archive.testzip() is None, "ZIP member CRC failed: archive may be corrupt.")
    inventory = pd.DataFrame([
        {"member": item.filename, "bytes": item.file_size, "compressed_bytes": item.compress_size}
        for item in archive.infolist() if not item.is_dir()
    ])
    expected_members = {f"{LAYER}{suffix}" for suffix in [".shp", ".shx", ".dbf", ".prj", ".cpg", ".xml"]}
    require(set(inventory["member"]) == expected_members, "Archive contents differ from the inspected release.")
    metadata_root = ET.fromstring(archive.read(f"{LAYER}.xml"))
    metadata_text = [node.text.strip() for node in metadata_root.iter() if node.text and node.text.strip()]
    licence_lines = [text for text in metadata_text if "Creative Commons" in text]
    require(bool(licence_lines), "Expected licence text missing from ABS metadata.")
print("Archive members:", len(inventory))
display(inventory)
print("Licence from archive metadata:", licence_lines[0])
print("Source ZIP bytes:", BOUNDARY_ZIP.stat().st_size)
print("Source ZIP SHA-256:", fingerprint(BOUNDARY_ZIP))

## 4. Read the boundary layer and audit Victoria

`/vsizip/` is GDAL's way to read inside a ZIP; `as_posix()` handles the Windows drive path without
manually extracting or renaming the archive. We do not use a bounding box to define Melbourne.
First filter **Victoria (`STE_CODE21 == "2"`)**, then match the rental source's 31 areas.

**Expect 82 Victorian records, not 82 councils.** This file contains 79 council-area records,
one `Unincorporated Vic` polygon, and two non-spatial categories. The two null geometries are expected
for `No usual address (Vic.)` and `Migratory - Offshore - Shipping (Vic.)`.

At national level the 567 records include 19 non-spatial categories. A null shape is not automatically
a corrupt dataset. It must nevertheless never be used as the polygon of a rental LGA.

In [ ]:
# GDAL VSI paths work with Windows drive paths and spaces as well as POSIX paths.
BOUNDARY_URI = f"/vsizip/{BOUNDARY_ZIP.resolve().as_posix()}/{LAYER}.shp"
layer_inventory = pyogrio.list_layers(BOUNDARY_URI)
require(layer_inventory[0, 0] == LAYER, "Unexpected boundary layer.")
australia = gpd.read_file(BOUNDARY_URI, engine="pyogrio")
expected_fields = ["LGA_CODE25", "LGA_NAME25", "STE_CODE21", "STE_NAME21",
                   "AUS_CODE21", "AUS_NAME21", "AREASQKM", "geometry"]
require(australia.columns.tolist() == expected_fields, "Boundary field layout changed.")
require(australia.crs is not None and australia.crs.to_epsg() == 7844, "Expected GDA2020 / EPSG:7844.")
require(len(australia) == 567 and int(australia.geometry.isna().sum()) == 19,
        "National boundary inventory changed.")
for column in ["LGA_CODE25", "LGA_NAME25", "STE_CODE21", "STE_NAME21"]:
    australia[column] = australia[column].astype("string")
require(australia["LGA_CODE25"].is_unique, "Duplicate national LGA code.")
victoria = australia.loc[australia["STE_CODE21"].eq("2")].copy()
require(len(victoria) == 82 and victoria["STE_NAME21"].eq("Victoria").all(), "Unexpected Victoria inventory.")
nonspatial = victoria.loc[victoria.geometry.isna(), ["LGA_CODE25", "LGA_NAME25"]].copy()
require(set(nonspatial["LGA_CODE25"]) == {"29499", "29799"}, "Unexpected missing Victorian geometries.")
EXCLUDED_VIC_CODES = {"29399", "29499", "29799"}
excluded_victoria = victoria.loc[victoria["LGA_CODE25"].isin(EXCLUDED_VIC_CODES)].copy()
excluded_victoria["exclusion_reason"] = excluded_victoria["LGA_CODE25"].map({
    "29399": "Unincorporated area; not one of the 79 council-area records",
    "29499": "Non-spatial statistical category: no usual address",
    "29799": "Non-spatial statistical category: migratory/offshore/shipping",
})
vic_councils = victoria.loc[~victoria["LGA_CODE25"].isin(EXCLUDED_VIC_CODES)].copy()
require(len(vic_councils) == 79, "Expected 79 Victorian council-area records.")
require(vic_councils.geometry.notna().all() and not vic_councils.geometry.is_empty.any(), "Missing council geometry.")
require(vic_councils.geometry.is_valid.all(), "Invalid council geometry; investigate instead of silently repairing.")
require(vic_councils.geom_type.isin(["Polygon", "MultiPolygon"]).all(), "Non-polygon council geometry.")
require(vic_councils["LGA_CODE25"].is_unique and vic_councils["LGA_NAME25"].is_unique, "Ambiguous Victorian identifiers.")

boundary_audit = pd.DataFrame([
    {"scope": name, "records": len(frame), "null_geometry": int(frame.geometry.isna().sum()),
     "empty_geometry": int(frame.geometry.is_empty.sum()),
     "invalid_non_null_geometry": int((frame.geometry.notna() & ~frame.geometry.is_valid).sum())}
    for name, frame in [("Australia (all codes)", australia), ("Victoria (all codes)", victoria),
                        ("Victoria (council areas only)", vic_councils)]
])
print("Boundary CRS:", australia.crs)
display(boundary_audit)
print("Explicitly excluded Victorian categories:")
display(excluded_victoria[["LGA_CODE25", "LGA_NAME25", "exclusion_reason"]])
print("Actual name and code fields:")
display(vic_councils[["LGA_CODE25", "LGA_NAME25", "STE_CODE21"]].head())

## 5. Try exact names first and show every mismatch

This is an **attribute join**: compare area-name fields, then use reviewed LGA codes.
A **spatial join** (for example, which LGA contains a bus stop) is a later operation.

Do not throw away unmatched rows with an inner join. We use a left join and a merge indicator
so every rental area remains visible. The initial exact-name match should find 28 of 31 areas.

In [ ]:
source_inventory = latest[["rental_area_name", "source_region"]].copy()
name_lookup = vic_councils[["LGA_CODE25", "LGA_NAME25"]].rename(columns={
    "LGA_CODE25": "lga_code_2025", "LGA_NAME25": "abs_lga_name",
})
first_pass = source_inventory.merge(
    name_lookup, left_on="rental_area_name", right_on="abs_lga_name",
    how="left", validate="one_to_one", indicator=True,
)
unmatched_before = first_pass.loc[first_pass["_merge"].eq("left_only"),
                                  ["rental_area_name", "source_region"]].copy()
exact_match_count = int(first_pass["_merge"].eq("both").sum())
require(exact_match_count == 28 and len(unmatched_before) == 3, "Exact-name inventory differs from the review.")
print("Exact matches:", exact_match_count, "/", len(source_inventory))
print("Name differences requiring explicit review:")
display(unmatched_before.reset_index(drop=True))

## 6. Record the three reviewed name associations

These mappings are based on the two uploaded source inventories, the state identifier, and ABS's naming
rule that adds state suffixes for names shared across states. They are **not fuzzy matches**.

| Rental name retained | ABS 2025 name retained | ABS code |
|---|---|---|
| Bayside | Bayside (Vic.) | 20910 |
| Kingston | Kingston (Vic.) | 23430 |
| Mornington Penin'a | Mornington Peninsula | 25340 |

A **crosswalk** is simply this kind of translation table. The resulting full crosswalk records all
31 associations, their method and notes. Review its output. We do not rename the original rental data.
Names/codes are reviewed associations; identical boundary definitions across publishers are not demonstrated.

In [ ]:
reviewed_aliases = pd.DataFrame([
    {"rental_area_name": "Bayside", "lga_code_2025": "20910", "abs_lga_name": "Bayside (Vic.)",
     "match_note": "Victoria-only lookup; ABS adds a state suffix to distinguish the name."},
    {"rental_area_name": "Kingston", "lga_code_2025": "23430", "abs_lga_name": "Kingston (Vic.)",
     "match_note": "Victoria-only lookup; ABS adds a state suffix to distinguish the name."},
    {"rental_area_name": "Mornington Penin'a", "lga_code_2025": "25340", "abs_lga_name": "Mornington Peninsula",
     "match_note": "Reviewed expansion of the rental source's abbreviated name; Victoria code verified."},
]).astype("string")
require(set(reviewed_aliases["rental_area_name"]) == set(unmatched_before["rental_area_name"]),
        "The alias table must cover exactly the three observed mismatches.")

def build_crosswalk(source_names, lookup, aliases):
    require(source_names["rental_area_name"].notna().all() and source_names["rental_area_name"].is_unique,
            "Rental area names must be complete and unique.")
    require(lookup["lga_code_2025"].is_unique and lookup["abs_lga_name"].is_unique,
            "Boundary lookup is ambiguous.")
    require(aliases["rental_area_name"].is_unique, "Duplicate reviewed alias.")
    by_name = lookup.set_index("abs_lga_name")["lga_code_2025"].to_dict()
    by_code = lookup.set_index("lga_code_2025")["abs_lga_name"].to_dict()
    alias_by_source = aliases.set_index("rental_area_name").to_dict(orient="index")
    rows = []
    for row in source_names.itertuples(index=False):
        name = row.rental_area_name
        if name in by_name:
            code_value, official, method, note = by_name[name], name, "exact_name", "Exact name within Victoria."
        else:
            require(name in alias_by_source, f"Unreviewed source name: {name}")
            alias = alias_by_source[name]
            code_value, official = alias["lga_code_2025"], alias["abs_lga_name"]
            require(by_code.get(code_value) == official, f"Reviewed code/name pair no longer exists: {name}")
            method, note = "reviewed_alias", alias["match_note"]
        rows.append({"rental_area_name": name, "source_region": row.source_region,
                     "lga_code_2025": code_value, "abs_lga_name": official,
                     "boundary_year": BOUNDARY_YEAR, "match_method": method, "match_note": note})
    result = pd.DataFrame(rows).sort_values("rental_area_name", kind="stable").reset_index(drop=True)
    for column in result.columns.difference(["boundary_year"]):
        result[column] = result[column].astype("string")
    require(result["lga_code_2025"].notna().all() and result["lga_code_2025"].is_unique,
            "Multiple rental areas map to the same code, or a code is missing.")
    require(result["lga_code_2025"].str.fullmatch(r"2[0-9]{4}").all(), "Invalid Victorian LGA code.")
    return result

crosswalk = build_crosswalk(source_inventory, name_lookup, reviewed_aliases)
require(len(crosswalk) == 31, "Incomplete crosswalk.")
require(crosswalk["match_method"].value_counts().to_dict() == {"exact_name": 28, "reviewed_alias": 3},
        "Unexpected match-method totals.")
print("Matched rental areas:", len(crosswalk), "/", len(latest))
display(crosswalk[["rental_area_name", "abs_lga_name", "lga_code_2025", "match_method"]])

## 7. Attach shapes without altering rents

Once the crosswalk is reviewed, **codes** join to polygons. We keep the GeoDataFrame on the left of the
merge so it retains its geometry/CRS. `validate="one_to_one"` rejects accidental row multiplication.
The comparison below checks all 14 original rental fields, not just the displayed price.

The native geometry layer retains source coordinates in **GDA2020 / EPSG:7844**.
No geometry is clipped, dissolved, simplified, or repaired. Polygon and MultiPolygon both describe area shapes;
a MultiPolygon can represent a single LGA with multiple disconnected pieces.

In [ ]:
matched_codes = set(crosswalk["lga_code_2025"])
metro_native = vic_councils.loc[vic_councils["LGA_CODE25"].isin(matched_codes),
                              ["LGA_CODE25", "LGA_NAME25", "STE_CODE21", "STE_NAME21", "AREASQKM", "geometry"]].copy()
metro_native = metro_native.rename(columns={
    "LGA_CODE25": "lga_code_2025", "LGA_NAME25": "abs_lga_name",
    "STE_CODE21": "state_code", "STE_NAME21": "state_name", "AREASQKM": "abs_area_sqkm",
})
metro_native["boundary_year"] = BOUNDARY_YEAR
metro_native["boundary_source"] = "Australian Bureau of Statistics; ASGS Edition 3 LGA 2025; CC BY 4.0"
metro_native = metro_native.sort_values("lga_code_2025", kind="stable").reset_index(drop=True)
require(len(metro_native) == 31, "A mapped code has no boundary.")

rent_with_codes = latest.merge(
    crosswalk[["rental_area_name", "lga_code_2025", "match_method", "match_note"]],
    on="rental_area_name", how="left", validate="one_to_one",
)
mapped = metro_native.merge(rent_with_codes, on="lga_code_2025", how="left", validate="one_to_one")
mapped = mapped.sort_values("rental_area_name", kind="stable").reset_index(drop=True)

def verify_map(frame, source_rents, mapping, source_geometry):
    require(isinstance(frame, gpd.GeoDataFrame) and frame.crs == source_geometry.crs, "Map lost its native CRS.")
    require(len(frame) == len(source_rents) == len(mapping), "Join lost or added rows.")
    require(frame["lga_code_2025"].is_unique and frame["rental_area_name"].is_unique, "Duplicate map feature.")
    require(set(frame["rental_area_name"]) == set(source_rents["rental_area_name"]), "Rental area set changed.")
    require(set(frame["lga_code_2025"]) == set(mapping["lga_code_2025"]), "Mapped code set changed.")
    require(frame.geometry.notna().all() and not frame.geometry.is_empty.any(), "Missing or empty mapped geometry.")
    require(frame.geometry.is_valid.all(), "Invalid mapped geometry.")
    require(frame.geom_type.isin(["Polygon", "MultiPolygon"]).all(), "Map includes a non-area feature.")
    observed_pairs = set(zip(frame["rental_area_name"], frame["lga_code_2025"], frame["abs_lga_name"]))
    expected_pairs = set(zip(mapping["rental_area_name"], mapping["lga_code_2025"], mapping["abs_lga_name"]))
    require(observed_pairs == expected_pairs, "Name/code associations differ from the reviewed crosswalk.")
    actual = pd.DataFrame(frame[COLUMN_ORDER]).sort_values(KEY_COLUMNS).reset_index(drop=True)
    expected = source_rents[COLUMN_ORDER].sort_values(KEY_COLUMNS).reset_index(drop=True)
    pd.testing.assert_frame_equal(actual, expected, check_exact=True)
    geometries_by_code = source_geometry.set_index("lga_code_2025").geometry
    require(all(shape.equals_exact(geometries_by_code.loc[code_value], 0.0)
                for code_value, shape in zip(frame["lga_code_2025"], frame.geometry)),
            "A mapped geometry differs from the shape belonging to its code.")

verify_map(mapped, latest, crosswalk, metro_native)
print("GEOGRAPHY JOIN CHECKPOINT PASSED")
print("Matched areas: 31/31 | Duplicate codes: 0 | Missing or invalid mapped geometries: 0")
print("All 14 rental fields preserved: True")
print("Native boundary shapes preserved: True")
display(mapped[["rental_area_name", "abs_lga_name", "lga_code_2025", "quarter", "median_weekly_rent", "published_count"]])

## 8. Test that the safeguards actually reject bad examples

These are copies held in memory only, not modifications to the source data.
We deliberately try an unknown name, duplicate lookup, swapped name/code association, missing geometry,
and altered rent. Each must be rejected. These five notebook self-checks supplement the existing pipeline tests;
they are not five extra pytest cases yet.

In [ ]:
def expect_rejected(label, action):
    try:
        action()
    except (ValueError, AssertionError, pd.errors.MergeError):
        print("Rejected as expected:", label)
        return
    raise AssertionError(f"Safeguard did not reject: {label}")

bad_names = source_inventory.copy(deep=True)
bad_names.loc[bad_names.index[0], "rental_area_name"] = "Unknown fictional area"
expect_rejected("unreviewed area name", lambda: build_crosswalk(bad_names, name_lookup, reviewed_aliases))
ambiguous_lookup = pd.concat([name_lookup, name_lookup.iloc[[0]]], ignore_index=True)
expect_rejected("duplicate boundary lookup", lambda: build_crosswalk(source_inventory, ambiguous_lookup, reviewed_aliases))
wrong_code = mapped.copy(deep=True)
wrong_code.loc[[0, 1], "lga_code_2025"] = wrong_code.loc[[1, 0], "lga_code_2025"].to_numpy()
expect_rejected("swapped code associations", lambda: verify_map(wrong_code, latest, crosswalk, metro_native))
missing_shape = mapped.copy(deep=True)
missing_shape.at[0, "geometry"] = None
expect_rejected("missing geometry", lambda: verify_map(missing_shape, latest, crosswalk, metro_native))
altered_rent = mapped.copy(deep=True)
altered_rent.at[0, "median_weekly_rent"] += 1
expect_rejected("altered rent value", lambda: verify_map(altered_rent, latest, crosswalk, metro_native))
verify_map(mapped, latest, crosswalk, metro_native)
print("Five join self-checks passed; the real mapped data still passes.")

## 9. Prepare the browser coordinate system

**CRS** tells software how to interpret coordinates. The original shapes are GDA2020 (EPSG:7844).
For the web map we transform a **copy** into WGS84 longitude/latitude (EPSG:4326) with `to_crs()`.
`set_crs()` only labels coordinates; it is not a substitute for transforming them.

This is a visualisation transform, not a survey-grade accuracy claim. We calculate neither distances nor areas
in geographic degrees. Keep the native GeoPackage for future spatial processing; a projected metre-based CRS
will be chosen for distance/catchment work later.

No simplification is applied. The resulting map file is larger, but retains all boundary detail.

In [ ]:
web_map = mapped.to_crs("EPSG:4326")
require(web_map.crs.to_epsg() == 4326, "Web transform failed.")
require(web_map.geometry.is_valid.all(), "Geometry invalid after coordinate transform.")
west, south, east, north = web_map.total_bounds
require(144 < west < east < 147 and -39 < south < north < -37,
        "Map bounds are implausible for the selected Melbourne areas; check the CRS.")

# ISO text is used only in the exported GeoJSON copy, not in the cleaned rental table.
geojson_frame = web_map.copy(deep=True)
geojson_frame["quarter_end"] = geojson_frame["quarter_end"].dt.strftime("%Y-%m-%d")
map_geojson = json.loads(geojson_frame.to_json(drop_id=True, na="null"))
for feature in map_geojson["features"]:
    feature["id"] = feature["properties"]["lga_code_2025"]
require(len(map_geojson["features"]) == 31, "Wrong number of web features.")
require({feature["id"] for feature in map_geojson["features"]} == matched_codes, "GeoJSON ID mismatch.")
require(all(feature["geometry"]["type"] in {"Polygon", "MultiPolygon"}
            for feature in map_geojson["features"]), "Invalid GeoJSON geometry type.")
print("Native CRS:", metro_native.crs, "| Web CRS:", web_map.crs)
print("Map feature IDs:", len(map_geojson["features"]), "unique 2025 LGA codes")
print("Web bounds [west, south, east, north]:", [round(float(x), 5) for x in [west, south, east, north]])

## 10. Build the first interactive rent map

This is a **choropleth**: polygons coloured by a numeric measure. Use the colour legend to interpret
**published median AUD per week**. A larger coloured polygon is a larger geographic area, not a higher price
or a larger number of rentals. Prices still vary within each LGA. Hover for its exact name, code, count, quarter
and original Excel source cells. Zoom in to inspect the small inner-city LGAs.

The default map uses `white-bg`: no background-map service, key, or external tiles are required. Blank space
outside the selected polygons is simply unplotted space (including water and out-of-scope areas), **not $0 rent**.
The three previously used audit LGAs have optional point markers to help you orient yourself. These are
interior label anchors, not property locations, stations, or exact city centres.

For optional street/place context after the first successful run, change `USE_ONLINE_BASEMAP` to True and
rerun sections 10–12. That mode requests CARTO/OpenStreetMap tiles over the internet and discloses the displayed
map extent to that provider. The initial offline-style map makes no such requests. No data is geocoded.

In [ ]:
USE_ONLINE_BASEMAP = False
MAP_STYLE = "carto-positron" if USE_ONLINE_BASEMAP else "white-bg"

hover_columns = ["rental_area_name", "abs_lga_name", "quarter", "property_type", "published_count",
                 "source_sheet", "source_count_cell", "source_median_cell", "match_method"]
custom = mapped[hover_columns].copy()
custom["published_count"] = custom["published_count"].astype(int)
fig = go.Figure(go.Choroplethmap(
    geojson=map_geojson,
    locations=mapped["lga_code_2025"].tolist(),
    featureidkey="properties.lga_code_2025",
    z=mapped["median_weekly_rent"].astype(float).tolist(),
    zmin=float(mapped["median_weekly_rent"].min()),
    zmax=float(mapped["median_weekly_rent"].max()),
    customdata=custom.to_numpy(),
    marker={"line": {"width": 1}},
    colorbar={"title": {"text": "Median rent<br>AUD / week"}, "tickprefix": "$", "len": 0.70},
    hovertemplate=(
        "<b>%{customdata[1]}</b><br>Rental-source name: %{customdata[0]}<br>"
        "ABS LGA code: %{location}<br>Quarter: %{customdata[2]} | %{customdata[3]}<br>"
        "Median: $%{z:,.0f} per week<br>Published count: %{customdata[4]:,}<br>"
        "Excel: %{customdata[5]}!%{customdata[6]} / %{customdata[7]}<br>"
        "Match: %{customdata[8]}<extra></extra>"
    ), name="Published median rent",
))
# Marker positions are derived from each source polygon, not guessed coordinates.
anchors = metro_native.loc[metro_native["abs_lga_name"].isin(["Melbourne", "Monash", "Whitehorse"])].copy()
anchors["geometry"] = anchors.geometry.representative_point()
anchors = anchors.to_crs(4326)
fig.add_trace(go.Scattermap(
    lon=anchors.geometry.x.tolist(), lat=anchors.geometry.y.tolist(),
    mode="markers", marker={"size": 9}, name="Three audit-example LGAs",
    text=anchors["abs_lga_name"].tolist(),
    hovertemplate="<b>%{text}</b><br>Interior label anchor, not a rental address<extra></extra>",
    showlegend=True,
))
fig.update_layout(
    title={"text": "HomeReach Melbourne | Two-bedroom flat rents<br>"
                   "<sup>September quarter 2025 · 31 source-group metropolitan LGAs · ABS 2025 boundaries</sup>",
           "x": 0.02},
    map={"style": MAP_STYLE, "center": {"lon": float((west + east) / 2), "lat": float((south + north) / 2)},
         "zoom": 7.8, "bearing": 0, "pitch": 0},
    height=800, margin={"l": 12, "r": 12, "t": 95, "b": 100},
    legend={"orientation": "h", "x": 0, "y": -0.03},
    annotations=[{"text": "Rents: DFFH · Boundaries: ABS ASGS LGA 2025 (CC BY 4.0)<br>"
                          "Historical area medians, not live listings. Statistical boundaries, not legal definitions.",
                  "xref": "paper", "yref": "paper", "x": 0, "y": -0.09,
                  "xanchor": "left", "yanchor": "top", "showarrow": False}],
)
fig.show(config={"responsive": True, "scrollZoom": True, "displaylogo": False})
print("Map rent range: $%g–$%g per week" % (mapped["median_weekly_rent"].min(), mapped["median_weekly_rent"].max()))
print("Basemap:", "online CARTO tiles" if USE_ONLINE_BASEMAP else "white background; no external basemap")

## 11. Export and read back the outputs

The **GeoPackage** stores the 31 native boundary shapes and identifiers, without duplicating the whole
historical rental table. The **GeoJSON** stores the 2025Q3 joined map in web coordinates. CSVs record the
crosswalk and audits. The HTML map embeds its data and Plotly JavaScript; with the default white background
it can be opened without a tile service. Its file may be several megabytes because the full shapes are retained.

We compare values and shapes after reloading. For GeoJSON coordinate read-back, a tolerance of `1e-10` degrees
only allows numeric serialisation rounding; it is **not** a claim about real-world boundary accuracy.
Files are staged before replacement. Close generated outputs in GIS/Excel before rerunning on Windows.
Replacements are per-file, not a multi-file database transaction; do not run simultaneous writers.

In [ ]:
GEO_DIR = ROOT / "data/interim/geography"
PROCESSED_GEO = ROOT / "data/processed/geography"
GPKG_PATH = PROCESSED_GEO / "metro_lga_2025.gpkg"
GEOJSON_PATH = GEO_DIR / "rent_map_2025q3.geojson"
HTML_PATH = GEO_DIR / "rent_map_2025q3.html"
GEO_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_GEO.mkdir(parents=True, exist_ok=True)

def check_native_roundtrip(actual, expected):
    require(actual.crs == expected.crs and len(actual) == len(expected), "Native geometry read-back changed CRS/count.")
    actual = actual.sort_values("lga_code_2025").reset_index(drop=True)
    expected = expected.sort_values("lga_code_2025").reset_index(drop=True)
    attrs = [column for column in expected.columns if column != "geometry"]
    require(set(actual.columns) == set(expected.columns), "Native geometry read-back changed columns.")
    for column in attrs:
        require(actual[column].tolist() == expected[column].tolist(), f"GeoPackage attribute changed: {column}")
    require(all(a.equals_exact(b, 0.0) for a, b in zip(actual.geometry, expected.geometry)),
            "Native geometry changed during GeoPackage export.")

tables = {
    "lga_crosswalk_2025.csv": crosswalk,
    "unmatched_before_review.csv": unmatched_before.reset_index(drop=True),
    "unmatched_after_review.csv": crosswalk.iloc[0:0][["rental_area_name", "lga_code_2025"]],
    "boundary_geometry_audit.csv": boundary_audit,
    "excluded_victorian_records.csv": pd.DataFrame(excluded_victoria[["LGA_CODE25", "LGA_NAME25", "exclusion_reason"]]),
    "latest_rent_by_lga_code.csv": pd.DataFrame(mapped.drop(columns="geometry")),
}
output_hashes = {}
with TemporaryDirectory(prefix="geography_stage_", dir=GEO_DIR) as temporary:
    stage = Path(temporary)
    staged_gpkg = stage / GPKG_PATH.name
    # GEOMETRY permits Polygon and MultiPolygon without changing their structure.
    pyogrio.write_dataframe(metro_native, staged_gpkg, layer="metro_lga_2025", driver="GPKG",
                            geometry_type="Unknown", promote_to_multi=False)
    native_back = gpd.read_file(staged_gpkg, layer="metro_lga_2025", engine="pyogrio")
    check_native_roundtrip(native_back, metro_native)
    staged_geojson = stage / GEOJSON_PATH.name
    staged_geojson.write_text(json.dumps(map_geojson, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    loaded_geojson = json.loads(staged_geojson.read_text(encoding="utf-8"))
    require(loaded_geojson == map_geojson, "GeoJSON properties or coordinates changed on disk.")
    web_back = gpd.read_file(staged_geojson, engine="pyogrio").sort_values("lga_code_2025").reset_index(drop=True)
    web_expected = web_map.sort_values("lga_code_2025").reset_index(drop=True)
    require(web_back.crs.to_epsg() == 4326 and len(web_back) == 31, "GeoJSON spatial read-back failed.")
    require(all(a.equals_exact(b, 1e-10) for a, b in zip(web_back.geometry, web_expected.geometry)),
            "Web geometry read-back differs.")
    require(web_back["lga_code_2025"].astype(str).tolist() == web_expected["lga_code_2025"].tolist(), "GeoJSON code mismatch.")
    for column in ["median_weekly_rent", "published_count"]:
        require(np.array_equal(web_back[column].to_numpy(dtype=float), web_expected[column].to_numpy(dtype=float)),
                f"GeoJSON changed {column}.")
    staged_html = stage / HTML_PATH.name
    fig.write_html(staged_html, include_plotlyjs=True, full_html=True,
                   config={"responsive": True, "scrollZoom": True, "displaylogo": False})
    require(staged_html.stat().st_size > 100_000, "HTML export is unexpectedly small.")
    staged_outputs = [(staged_gpkg, GPKG_PATH), (staged_geojson, GEOJSON_PATH), (staged_html, HTML_PATH)]
    for name, table in tables.items():
        path = stage / name
        table.to_csv(path, index=False)
        # Compare a canonical text view to avoid dtype guesses in mixed audit tables.
        reloaded = pd.read_csv(path, dtype="string", keep_default_na=False)
        expected_text = table.astype("string").fillna("").reset_index(drop=True)
        pd.testing.assert_frame_equal(reloaded, expected_text, check_exact=True)
        staged_outputs.append((path, GEO_DIR / name))
    pd.testing.assert_frame_equal(panel, panel_before, check_exact=True)
    pd.testing.assert_frame_equal(latest, latest_before, check_exact=True)
    require(all(fingerprint(path) == saved_hash for path, saved_hash in input_hashes.items()),
            "An input changed during this notebook run.")
    for staged, destination in staged_outputs:
        staged.replace(destination)
        output_hashes[str(destination.relative_to(ROOT))] = fingerprint(destination)

print("GeoPackage read-back: matched")
print("GeoJSON properties and geometry read-back: matched")
print("Crosswalk and audit CSV read-back: matched")
print("Saved interactive map:", HTML_PATH.relative_to(ROOT))

## 12. Record the audit and final checkpoint

The generated report below includes all 31 mappings and the three naming decisions, the ZIP fingerprint,
CRS, source/licence references, output checks, and interpretation limitations. Rerunning this cell replaces
that report, so keep personal reflections in your progress notes instead.

The registry is intentionally left unchanged until you have reviewed the map and crosswalk locally.
Saving files is not proof that a browser rendered the map correctly: open the HTML and inspect its labels/hover.

In [ ]:
REPORT_PATH = ROOT / "docs/lga_geography_audit.md"
MANIFEST_PATH = GEO_DIR / "geography_manifest.json"
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
rows = [
    "# LGA geography audit and first rent map", "",
    "## Scope and source", "",
    f"Rent: {MAP_QUARTER}; {spec['property_type']}; 31 LGAs in DFFH's three metropolitan groups.",
    "Map matches one quarter only. The 3,286-row historical rental panel is unchanged.",
    f"Boundary archive: {BOUNDARY_ZIP.name}; {BOUNDARY_ZIP.stat().st_size:,} bytes.",
    f"Boundary SHA-256: `{EXPECTED_ZIP_SHA256}`.",
    f"ABS source: {ABS_DOWNLOAD}", f"ABS definitions: {ABS_PAGE}",
    f"Rental definitions: {spec['publisher_page']}",
    f"Licence text in boundary metadata: {licence_lines[0]}", "",
    "## Boundary inventory", "",
    "567 national records: 548 spatial records and 19 non-spatial codes.",
    "82 Victorian records: 79 council areas, Unincorporated Vic (29399), and two non-spatial categories.",
    "No usual address (Vic.) (29499) and Migratory - Offshore - Shipping (Vic.) (29799) have null geometry.",
    "The three special Victorian records are excluded from the council lookup, not converted into invented shapes.",
    "Native CRS: GDA2020 / EPSG:7844. Web-map copy: WGS84 / EPSG:4326 using to_crs().",
    "Mapped source geometries: 30 Polygon records and 1 MultiPolygon record. All 31 are non-null, non-empty and valid.",
    "No clipping, dissolving, simplification, buffering or automatic repair is applied.", "",
    "## Reviewed matching", "",
    "28 exact name matches within Victoria plus 3 explicitly reviewed aliases; 31/31 matched.",
    "No fuzzy matching. Both publishers' names and the 2025 ABS code are retained.",
    "Name/code matching is a reviewed association, not proof of exactly equivalent geographic definitions.", "",
    "| Rental-source name | ABS 2025 name | Code | Method |",
    "|---|---|---|---|",
]
for row in crosswalk.itertuples(index=False):
    rows.append(f"| {row.rental_area_name} | {row.abs_lga_name} | {row.lga_code_2025} | {row.match_method} |")
rows += ["", "## Validation", "",
    "All 14 original rental fields and all 31 native shapes matched after the attribute join.",
    "Five rejection self-checks passed: unknown name, duplicate lookup, swapped code, missing shape and altered rent.",
    "GeoPackage shapes/attributes, GeoJSON values/shapes and six CSV tables passed read-back checks.",
    "Native inputs, existing notebooks, registry and cleaned rental data stayed unchanged.", "",
    "## Outputs", "",
    "- `data/processed/geography/metro_lga_2025.gpkg`: native boundary dimension with 31 features.",
    "- `data/interim/geography/lga_crosswalk_2025.csv`: full reviewed name/code mapping.",
    "- `data/interim/geography/rent_map_2025q3.geojson`: joined map feature data in WGS84.",
    "- `data/interim/geography/rent_map_2025q3.html`: interactive map.",
    "- Other CSVs retain before/after match reports, exclusions, geometry audit and latest attributed rows.", "",
    "## Interpretation limits", "",
    "ABS polygons are statistical approximations, not legal boundaries. DFFH group scope is preserved.",
    "Colour represents nominal median AUD/week, not suitability, income-based affordability or available stock.",
    "Larger polygons indicate larger geographic footprints, not more expensive rents or more observations.",
    "Prices vary inside an LGA. Blank space outside mapped polygons is not a zero-rent observation.",
    "Markers are interior label anchors, not rental-property locations or transport stops.",
    "These are historical 2025Q3 statistics, not live listings. Do not apply 2025 shapes to earlier years without review.",
    f"Map basemap mode: {MAP_STYLE}. Full boundary detail is retained.",
    "This is a descriptive prototype: no forecasts, routes, transport scores or causal spatial findings are produced.", "",
    "## References for implementation", "",
    "- https://geopandas.org/en/stable/docs/user_guide/io.html",
    "- https://geopandas.org/en/stable/docs/user_guide/mergingdata.html",
    "- https://geopandas.org/en/stable/docs/user_guide/projections.html",
    "- https://plotly.com/python/tile-map-layers/", "",
]
REPORT_PATH.write_text("\n".join(rows), encoding="utf-8")
manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "builder": "notebooks/04_lga_geography_and_rent_map.ipynb",
    "boundary_source": {"filename": BOUNDARY_ZIP.name, "sha256": EXPECTED_ZIP_SHA256,
                        "download": ABS_DOWNLOAD, "layer": LAYER, "vintage": BOUNDARY_YEAR,
                        "native_crs": "EPSG:7844", "licence_from_xml": licence_lines[0]},
    "rental_source": {"csv_sha256": fingerprint(INPUT), "source_xlsx_sha256": spec["sha256"],
                      "quarter": MAP_QUARTER, "property_type": spec["property_type"],
                      "scope": "DFFH three metropolitan groups; 31 individual LGA rows"},
    "quality": {"national_records": len(australia), "victorian_records": len(victoria),
                "victorian_council_records": len(vic_councils), "exact_matches": exact_match_count,
                "reviewed_alias_matches": 3, "mapped_features": len(mapped), "unmatched": 0,
                "duplicate_codes": 0, "missing_or_invalid_mapped_geometry": 0,
                "original_rental_fields_preserved": True, "join_self_checks_passed": 5,
                "gpkg_readback_matches": True, "geojson_readback_matches": True,
                "csv_readbacks_match": True, "inputs_unchanged": True},
    "web_map": {"crs": "EPSG:4326", "style": MAP_STYLE, "simplification": "none",
                "browser_visual_check": "Required on the user's computer; not inferred from file creation."},
    "versions": {"python": sys.version.split()[0], "pandas": pd.__version__, "geopandas": gpd.__version__,
                 "pyogrio": pyogrio.__version__, "shapely": shapely.__version__, "pyproj": pyproj.__version__,
                 "plotly": plotly.__version__},
    "input_sha256": {str(path.relative_to(ROOT)): value for path, value in input_hashes.items()},
    "output_sha256": {**output_hashes, str(REPORT_PATH.relative_to(ROOT)): fingerprint(REPORT_PATH)},
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
require(all(fingerprint(path) == saved_hash for path, saved_hash in input_hashes.items()), "An input changed.")
print("MAP CHECKPOINT PASSED")
print("Boundary records: 567 national / 82 Victorian / 79 Victorian council areas")
print("Rental quarter: 2025Q3 | Property type: 2 bedroom flats")
print("Exact-name matches: 28")
print("Reviewed aliases: 3")
print("Matched rental LGAs: 31/31")
print("Unmatched rental LGAs: 0")
print("Duplicate LGA codes: 0")
print("Missing or invalid mapped geometries: 0")
print("Original rental values and source references preserved: True")
print("Join self-checks: 5 passed")
print("GeoPackage, GeoJSON and CSV read-back: matched")
print("Input files unchanged: True")
print("Saved map:", HTML_PATH.relative_to(ROOT))
print("Saved report:", REPORT_PATH.relative_to(ROOT))

## Checkpoint: inspect the map before continuing

Save this notebook with **Ctrl+S**. Open `data/interim/geography/rent_map_2025q3.html` in Chrome/Edge.
You can double-click the HTML file in File Explorer. If the inline map is blank, try the exported HTML first;
the actual polygons/colourbar should be visible even though the default background has no streets or place labels.
No server is required. MapLibre uses WebGL; a browser graphics error is different from a failed data join.

Review the three alias rows in the crosswalk/report. On the map, verify these previously audited values:
**Melbourne $730 / count 3,209; Monash $600 / count 421; Whitehorse $600 / count 464** in 2025Q3.
They refer to the individual LGAs and two-bedroom-flat category, not whole-metropolitan medians.
The map should contain all 31 areas, including Bayside (Vic.), Kingston (Vic.), and Mornington Peninsula.

Explain one spatial observation in your own words, without inferring causes or making a rental recommendation.
A colour pattern can motivate a later question; it does not establish causality or statistical significance.

**Next checkpoint:** send the `MAP CHECKPOINT PASSED` output and a screenshot of the rendered map.
Do not upload the national ZIP or large generated maps to Git just to obtain a clean status.
We will review the results before promoting the crosswalk, updating the boundary registry and saving this milestone.